# Gold — RF-1 Risk matrix (Susceptibility × Consequence)

## 1. Read silver and score risk

`risk_score = S \u00d7 C` (1\u201325). Bands per the RF-1 5\u00d75 matrix:
Low 1\u20134 \u00b7 Moderate 5\u20139 \u00b7 High 10\u201319 \u00b7 Extreme 20\u201325.

In [ ]:
from pyspark.sql import functions as F

SILVER_WS    = "a7d0f907-bf14-4169-8d34-b8765824aa09"
SILVER_LH    = "7818d0c8-eacb-4599-a91c-68d795175857"
SILVER_ABFSS = f"abfss://{SILVER_WS}@onelake.dfs.fabric.microsoft.com/{SILVER_LH}/Tables"

silver = spark.read.format("delta").load(f"{SILVER_ABFSS}/silver_rf1_soil_susceptibility")

band_col = (
    F.when(F.col("risk_score") <= 4, "Low")
     .when(F.col("risk_score") <= 9, "Moderate")
     .when(F.col("risk_score") <= 19, "High")
     .otherwise("Extreme")
)

risk = (silver
    .withColumn("risk_score", F.col("s_rating") * F.col("c_rating"))
    .withColumn("risk_band", band_col))

print(f"Silver pixels read: {risk.count():,}")
risk.groupBy("risk_band").count().orderBy("risk_band").show()

## 2. Write the gold tables

- `gold_rf1_risk_pixels` \u2014 per-pixel risk score + band (with coordinates).
- `gold_rf1_risk_matrix` \u2014 the 5\u00d75 S\u00d7C grid (pixel count, mean risk, band).
- `gold_rf1_band_summary` \u2014 area (km\u00b2) and share per risk band.

In [ ]:
# --- 1) per-pixel risk ---
pixels = risk.select(
    "row", "col", "lon", "lat", "elevation_m", "slope_deg",
    "soil_soft", "soil_mapped",
    "s_rating", "c_rating", "risk_score", "risk_band",
    "aoi_name", "risk_factor", "ingested_at_utc",
)
pixels.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("gold_rf1_risk_pixels")

## 3. Visualise \u2014 risk matrix, map and band shares

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

BAND_COLORS = {"Low": "#2c7bb6", "Moderate": "#fdae61", "High": "#f46d43", "Extreme": "#a50026"}

mdf = spark.read.table("gold_rf1_risk_matrix").toPandas()
bdf = spark.read.table("gold_rf1_band_summary").toPandas()
pdf = spark.read.table("gold_rf1_risk_pixels").toPandas()

fig, ax = plt.subplots(1, 3, figsize=(18, 5.5))

# --- (a) 5x5 risk matrix coloured by band ---
def band_of(v):
    return "Low" if v <= 4 else "Moderate" if v <= 9 else "High" if v <= 19 else "Extreme"

cmap = ListedColormap([BAND_COLORS[b] for b in ["Low", "Moderate", "High", "Extreme"]])
norm = BoundaryNorm([1, 5, 10, 20, 26], cmap.N)
score_grid = np.zeros((5, 5), dtype=int)
count_grid = np.zeros((5, 5), dtype=int)
for _, r in mdf.iterrows():
    score_grid[int(r.c_rating) - 1, int(r.s_rating) - 1] = int(r.risk_score)
    count_grid[int(r.c_rating) - 1, int(r.s_rating) - 1] = int(r.pixel_count)

ax[0].imshow(score_grid, cmap=cmap, norm=norm, origin="lower", extent=[0.5, 5.5, 0.5, 5.5])
for s in range(1, 6):
    for c in range(1, 6):
        sc = s * c
        ax[0].text(s, c, f"{sc}\n({count_grid[c-1, s-1]:,})",
                   ha="center", va="center", fontsize=9,
                   color="white" if sc >= 10 else "black")
ax[0].set_xticks(range(1, 6)); ax[0].set_yticks(range(1, 6))
ax[0].set_xlabel("Susceptibility (S)"); ax[0].set_ylabel("Consequence (C)")
ax[0].set_title("RF-1 risk matrix  (S \u00d7 C \u2192 score, pixel count)")

# --- (b) spatial risk map ---
H = int(pdf["row"].max()) + 1
W = int(pdf["col"].max()) + 1
rmap = np.full((H, W), np.nan)
rmap[pdf["row"].values, pdf["col"].values] = pdf["risk_score"].values
im = ax[1].imshow(rmap, cmap="RdYlGn_r", vmin=1, vmax=25, origin="upper")
ax[1].set_title("Risk score over AOI (10 m)")
ax[1].set_xlabel("col"); ax[1].set_ylabel("row")
fig.colorbar(im, ax=ax[1], shrink=0.8, label="risk score")

# --- (c) band shares ---
order = ["Low", "Moderate", "High", "Extreme"]
bdf = bdf.set_index("risk_band").reindex(order).fillna(0).reset_index()
ax[2].bar(bdf["risk_band"], bdf["area_km2"],
          color=[BAND_COLORS[b] for b in bdf["risk_band"]])
for i, r in bdf.iterrows():
    ax[2].text(i, r.area_km2, f"{r.area_km2:.2f} km\u00b2\n{r.pct:.1f}%",
               ha="center", va="bottom", fontsize=9)
ax[2].set_ylabel("area (km\u00b2)"); ax[2].set_title("Area by risk band")

plt.tight_layout()
plt.show()